In [6]:
import re
import string
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [8]:
# ---- 1. Load brand mislabeling data ----
brand_df = pd.read_csv('../data/processed/amazon_brand_mislabel_pairs.csv')
print(f"Total rows: {len(brand_df)}")


Total rows: 500


In [9]:
# ============================================================
# PART A: Rule-based baseline
# ============================================================

def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def rule_based_check(title, manufacturer_shown):
    title_norm = normalize(title)
    manu_norm = normalize(manufacturer_shown)
    # Check if manufacturer (or its individual words) appear in the title
    return 1 if manu_norm in title_norm else 0  # 1 = looks correct, 0 = looks mislabeled

brand_df['rule_pred_correct'] = brand_df.apply(
    lambda row: rule_based_check(row['title'], row['manufacturer_shown']), axis=1
)
# Convert to same direction as is_mislabeled: rule says mislabeled (1) if NOT found in title (0)
brand_df['rule_pred_mislabeled'] = 1 - brand_df['rule_pred_correct']

p_rule = precision_score(brand_df['is_mislabeled'], brand_df['rule_pred_mislabeled'])
r_rule = recall_score(brand_df['is_mislabeled'], brand_df['rule_pred_mislabeled'])
f1_rule = f1_score(brand_df['is_mislabeled'], brand_df['rule_pred_mislabeled'])
acc_rule = accuracy_score(brand_df['is_mislabeled'], brand_df['rule_pred_mislabeled'])

print(f"\nRule-based baseline (manufacturer-in-title check):")
print(f"Precision: {p_rule:.3f}, Recall: {r_rule:.3f}, F1: {f1_rule:.3f}, Accuracy: {acc_rule:.3f}")


Rule-based baseline (manufacturer-in-title check):
Precision: 0.564, Recall: 1.000, F1: 0.721, Accuracy: 0.622


In [10]:
# ============================================================
# PART B: LLM zero-shot
# ============================================================

def classify_brand(title, manufacturer_shown, max_retries=3):
    prompt = f"""Product title: {title}
Listed manufacturer: {manufacturer_shown}

Based on the product title, is this manufacturer label likely correct or incorrect?
Respond with only one word: CORRECT or INCORRECT."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=5
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None, str(e)

def parse_brand_response(raw_response):
    if raw_response is None:
        return None
    r = raw_response.upper()
    if r == "CORRECT":
        return 0  # not mislabeled
    elif r == "INCORRECT":
        return 1  # mislabeled
    elif "INCORRECT" in r:
        return 1
    elif "CORRECT" in r:
        return 0
    return None

llm_results = []
errors = []
output_path = '../data/processed/amazon_brand_llm_zeroshot_preds.csv'

for idx, row in tqdm(brand_df.iterrows(), total=len(brand_df)):
    raw, err = classify_brand(row['title'], row['manufacturer_shown'])
    pred = parse_brand_response(raw)
    if err:
        errors.append({'idx': idx, 'id': row['id'], 'error': err})
    llm_results.append({
        'id': row['id'],
        'title': row['title'],
        'manufacturer_shown': row['manufacturer_shown'],
        'true_manufacturer': row['true_manufacturer'],
        'is_mislabeled': row['is_mislabeled'],
        'raw_response': raw,
        'llm_pred_mislabeled': pred
    })
    if (idx + 1) % 100 == 0:
        pd.DataFrame(llm_results).to_csv(output_path, index=False)

llm_df = pd.DataFrame(llm_results)
llm_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")
print(f"API errors: {len(errors)}")

valid = llm_df.dropna(subset=['llm_pred_mislabeled'])
p_llm = precision_score(valid['is_mislabeled'], valid['llm_pred_mislabeled'])
r_llm = recall_score(valid['is_mislabeled'], valid['llm_pred_mislabeled'])
f1_llm = f1_score(valid['is_mislabeled'], valid['llm_pred_mislabeled'])
acc_llm = accuracy_score(valid['is_mislabeled'], valid['llm_pred_mislabeled'])

print(f"\nLLM zero-shot (brand mislabeling):")
print(f"Precision: {p_llm:.3f}, Recall: {r_llm:.3f}, F1: {f1_llm:.3f}, Accuracy: {acc_llm:.3f}")

100%|██████████| 500/500 [06:22<00:00,  1.31it/s]


Saved to ../data/processed/amazon_brand_llm_zeroshot_preds.csv
API errors: 0

LLM zero-shot (brand mislabeling):
Precision: 0.827, Recall: 0.840, F1: 0.833, Accuracy: 0.836


In [11]:
# ============================================================
# PART C: Comparison table
# ============================================================

comparison = pd.DataFrame([
    {'Method': 'Rule-based (manufacturer-in-title)', 'Precision': round(p_rule, 3), 'Recall': round(r_rule, 3), 'F1': round(f1_rule, 3), 'Accuracy': round(acc_rule, 3)},
    {'Method': 'LLM zero-shot', 'Precision': round(p_llm, 3), 'Recall': round(r_llm, 3), 'F1': round(f1_llm, 3), 'Accuracy': round(acc_llm, 3)},
])
print("\n--- Brand Mislabeling Comparison Table ---")
print(comparison.to_string(index=False))
comparison.to_csv('../data/processed/amazon_brand_comparison_table.csv', index=False)


--- Brand Mislabeling Comparison Table ---
                            Method  Precision  Recall    F1  Accuracy
Rule-based (manufacturer-in-title)      0.564    1.00 0.721     0.622
                     LLM zero-shot      0.827    0.84 0.833     0.836
